<center><h1 style="background-color: #0B1F3A; color: white; padding:10px;"> HR Attrition Risk - Prediction</h1></center>

In [24]:
import pandas as pd
import joblib
import shap
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

#### Loading saved data from training

In [25]:
model = joblib.load("model.pkl")
mappings = joblib.load('mappings.pkl')
encoder = joblib.load('encoder.pkl')
feature_cols = joblib.load('feature_cols.pkl')

In [26]:
load_dotenv()

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

### Fetching employees data from Database

In [27]:
engine = create_engine(f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')
employee_df = pd.read_sql("SELECT * FROM employee", con=engine)

In [28]:
employee_df.shape

(1470, 32)

### Preprocessing

In [29]:
employee_df.head(2)

,employee_id,age,business_travel,department,environment_satisfaction,job_involvement,job_level,job_role,job_satisfaction,gender,...,years_since_last_promotion,training_times_last_year,work_life_balance,num_companies_worked,years_with_curr_manager,years_in_current_role,total_working_years,percent_salary_hike,overtime,replacement_cost
0,1,41,Travel_Rarely,Sales,2,3,2,Sales Executive,4,Female,...,0,0,1,8,5,4,8,11,Yes,71916.0
1,2,49,Travel_Frequently,Research & Development,3,2,2,Research Scientist,2,Male,...,1,3,3,1,7,7,10,23,No,61560.0


In [30]:
# converting employee_id as index
employee_df.set_index('employee_id',inplace=True)

In [31]:
employee_df.head(2)

,age,business_travel,department,environment_satisfaction,job_involvement,job_level,job_role,job_satisfaction,gender,marital_status,...,years_since_last_promotion,training_times_last_year,work_life_balance,num_companies_worked,years_with_curr_manager,years_in_current_role,total_working_years,percent_salary_hike,overtime,replacement_cost
employee_id,,,,,,,,,,,,,,,,,,,,,
1,41,Travel_Rarely,Sales,2,3,2,Sales Executive,4,Female,Single,...,0,0,1,8,5,4,8,11,Yes,71916.0
2,49,Travel_Frequently,Research & Development,3,2,2,Research Scientist,2,Male,Married,...,1,3,3,1,7,7,10,23,No,61560.0


In [32]:
# Encoding (Binary and ordinal)

for col, mapping in mappings.items():
    employee_df[col] = employee_df[col].map(mapping)

In [33]:
# one hot encoding
cols_to_transform = ['department', 'job_role', 'education_field', 'marital_status']
encoded_arr = encoder.transform(employee_df[cols_to_transform])
encoded_df = pd.DataFrame(encoded_arr, columns=encoder.get_feature_names_out(cols_to_transform), index=employee_df.index)
encoded_df = encoded_df.astype(int)
employee_df_final = pd.concat([employee_df.drop(columns = cols_to_transform), encoded_df], axis=1)

In [34]:
# matching columns and columns order
employee_df_final = employee_df_final[feature_cols]

In [35]:
employee_df_final.head(2)

,age,business_travel,daily_rate,distance_from_home,education,environment_satisfaction,gender,hourly_rate,job_involvement,job_satisfaction,...,job_role_Research Scientist,job_role_Sales Executive,job_role_Sales Representative,education_field_Life Sciences,education_field_Marketing,education_field_Medical,education_field_Other,education_field_Technical Degree,marital_status_Married,marital_status_Single
employee_id,,,,,,,,,,,,,,,,,,,,,
1,41,1,1102,1,2,2,1,94,3,4,...,0,1,0,1,0,0,0,0,0,1
2,49,2,279,8,1,3,0,61,2,2,...,1,0,0,1,0,0,0,0,1,0


### Predicting

In [36]:
y_probability = model.predict_proba(employee_df_final)[:,1]
# we need only probability for Class 1 (Attrition=Yes)
# threshold is 0.4

### SHAP explainer

In [37]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(employee_df_final)

In [38]:
shap_values_class1 = shap_values[:, :, 1]

### Top 3 Drivers

In [39]:
results = []
final_threshold = 0.4

for i in range(len(employee_df_final)):
    employee_shap_value = shap_values_class1[i]
    feature_impact = list(zip(employee_df_final.columns.tolist(), employee_shap_value))
    top_3 = sorted(feature_impact, key=lambda x: abs(x[1]), reverse=True)[:3]
    
    results.append({
        'employee_id': employee_df_final.index[i],
        'attrition_probability': round(y_probability[i], 3),
        'risk_label': 'High Risk' if y_probability[i] >= final_threshold else 'Low Risk',
        'top_driver_1': top_3[0][0],
        'top_driver_1_impact': round(top_3[0][1], 3),
        'top_driver_2': top_3[1][0],
        'top_driver_2_impact': round(top_3[1][1], 3),
        'top_driver_3': top_3[2][0],
        'top_driver_3_impact': round(top_3[2][1], 3),
    })

final_output = pd.DataFrame(results)
final_output.head(10)

,employee_id,attrition_probability,risk_label,top_driver_1,top_driver_1_impact,top_driver_2,top_driver_2_impact,top_driver_3,top_driver_3_impact
0,1,0.537,High Risk,overtime,0.063,stock_option_level,0.040,age,-0.039
1,2,0.287,Low Risk,age,-0.043,stock_option_level,-0.033,overtime,-0.033
2,4,0.642,High Risk,monthly_income,0.063,years_at_company,0.058,overtime,0.057
3,5,0.527,High Risk,overtime,0.072,stock_option_level,0.037,years_at_company,-0.036
4,7,0.510,High Risk,age,0.040,years_at_company,0.040,stock_option_level,-0.038
5,8,0.428,High Risk,overtime,-0.041,years_at_company,-0.040,stock_option_level,0.029
6,10,0.519,High Risk,years_at_company,0.065,age,-0.055,overtime,0.052
7,11,0.571,High Risk,years_at_company,0.093,stock_option_level,-0.041,age,0.040
8,12,0.329,Low Risk,age,-0.051,monthly_income,-0.037,overtime,-0.034
9,13,0.272,Low Risk,age,-0.039,stock_option_level,-0.038,years_at_company,-0.035


### Insert data to Database

In [40]:
final_output.to_sql(
    name='employee_prediction_history',
    con=engine,
    if_exists='append',
    index=False
)